In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

In [21]:
df = pd.read_csv('logistics_shipments_dataset 2.csv')

In [22]:
df = df.dropna(subset=['Origin_Warehouse','Destination','Carrier','Weight_kg','Transit_Days','Cost'])
df = df[df["Status"] == "Delivered"]
upper = df["Cost"].quantile(0.95)
df["Cost"] = df["Cost"].clip(upper=upper)

In [23]:
df.shape

(1612, 11)

In [24]:
df.describe()

,Weight_kg,Cost,Distance_miles,Transit_Days
count,1612.000000,1612.000000,1612.000000,1612.000000
mean,31.122519,197.885566,1291.426799,4.207816
std,138.736578,91.325602,691.202525,1.824616
min,0.000000,22.100000,101.000000,1.000000
25%,12.475000,119.812500,718.500000,3.000000
50%,20.600000,198.590000,1298.500000,4.000000
75%,34.100000,275.447500,1879.250000,5.000000
max,5404.200000,348.102000,2499.000000,12.000000


In [25]:
y = df['Cost']
y

0        67.460
1       268.850
2        74.350
3       187.040
4       120.010
         ...   
1995    217.780
1996    279.470
1997    250.320
1998    272.310
1999    348.102
Name: Cost, Length: 1612, dtype: float64

In [26]:
X = df[['Origin_Warehouse','Destination','Carrier','Weight_kg', 'Transit_Days']]
X

,Origin_Warehouse,Destination,Carrier,Weight_kg,Transit_Days
0,Warehouse_MIA,San Francisco,UPS,25.7,2
1,Warehouse_MIA,Atlanta,DHL,38.9,3
2,Warehouse_LA,Houston,DHL,37.2,2
3,Warehouse_BOS,Seattle,OnTrac,42.6,9
4,Warehouse_SF,Dallas,OnTrac,7.9,3
...,...,...,...,...,...
1995,Warehouse_BOS,San Francisco,FedEx,7.9,5
1996,Warehouse_HOU,Phoenix,UPS,36.5,5
1997,Warehouse_HOU,Portland,LaserShip,11.4,6
1998,Warehouse_SEA,Detroit,USPS,10.9,5


In [27]:
categorical = ['Origin_Warehouse','Destination', 'Carrier']
numerical = ['Weight_kg', 'Transit_Days']

In [28]:
preprocessor = ColumnTransformer(transformers = [('cat', OneHotEncoder(handle_unknown = 'ignore'), categorical),('num', 'passthrough', numerical)])
rf_model = RandomForestRegressor(n_estimators = 100, random_state=42)
pipeline = Pipeline(steps = [
        ("preprocessor", preprocessor),
        ("model", rf_model)
    ]
)

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))

MAE: 44.90692458677685
R²: 0.5737102443773208
